# Logistic Regression: Can baseline physical activity improve prediction of later chronic pain?
## A logistic regression case study using NLSY97

##### **Governing Question**
Among eligible NLSY97 participants, does adding self-reported physical-activity status at the 2015-round interview to age and general health improve logistic regression predictions of chronic pain reported at the 2019-round interview?

This question matches the five predictors currently declared in `configs/nlsy97.toml`. O*NET sitting is an **optional extension**: it is not a configured predictor, and its participant-level occupation mapping has not been validated. If that extension is completed, revise the question, title, feature contract, and comparisons together.

**Road map.** Define the investigation and data boundary; audit and explore eligible development data; explain logistic regression; connect student code and tests; specify evaluation; interpret results and failures; make a bounded product decision.

**Assignment references:** [assignment](../ASSIGNMENT.md), [case-study standard](../docs/NOTEBOOK_CASE_STUDY_STANDARD.md), [implementation guide](../STUDENT_IMPLEMENTATION_GUIDE.md), and [project checklist](../PROJECT_CHECKLIST.md). The numbered sections below follow the assignment's ten required questions. This notebook addresses one classification investigation within the larger final project.


## 1. Question, intended user, and investigation contract

**To complete in your own words:**

- Identify the intended reader, such as a student or researcher studying work, activity, and later health. What investigation would this model support, and what would a useful answer change?
- Define one observation as one eligible participant, linked across the baseline and outcome interviews. Distinguish the original cohort from the eventual modeling sample.
- Define the prediction moment as the baseline interview: only information already available then may be a predictor.
- Define the positive event as a **self-report of chronic pain due to illness or a medical condition during the preceding 30 days at follow-up**. Explain false-positive and false-negative predictions in this context.
- Explain why a probability model for a binary outcome addresses the question. State that this is predictive, observational work; the target does not establish new-onset pain or specifically back/neck pain.
- State the limits of use: this analysis does not diagnose, prescribe treatment, or establish that sitting or exercise causes pain. Describe the consequences of overstating the findings.

**Write here:** one opening paragraph explaining the question and its value, followed by a short paragraph specifying population, timing, and limits.


## 2. Data provenance and the data contract

### 2.1 Where the observations came from

**To complete:** link the [official NLSY97 overview](https://nlsinfo.org/content/cohorts/nlsy97) and [public data access page](https://nlsinfo.org/content/access-data-investigator/accessing-data-cohorts). Record the downloaded release, retrieval date, checksum or version, access/citation terms, observation unit, sampling design, geography, and retention limits in the [dataset card](../docs/DATASET_CARD.md). Do not assume that public access supplies an unrestricted redistribution license.

Explain that this is a fixed birth cohort followed over time. The nominal 2015 round contains 2015–16 interviews, and the nominal 2019 round contains 2019–20 interviews. Use actual interview dates when describing follow-up. Discuss self-report, attrition, and the population the retained sample can represent. Decide whether the estimand is predictive performance in the observed sample or a population-weighted quantity; justify any survey weighting separately.

### 2.2 The current configured variables

| Role | Column | Meaning / coding to verify against the codebook |
|---|---|---|
| Identifier | `PUBID__1997` | Participant ID; joins and partitions only, not a predictor |
| Baseline feature | `CV_AGE_INT_DATE__2015` | Age at baseline interview, in years |
| Baseline feature | `YHEA-100__2015` | General health: 1 excellent through 5 poor |
| Baseline feature | `YHEA-600AA__2015` | Vigorous-activity status |
| Baseline feature | `YHEA-600BA__2015` | Light/moderate-activity status |
| Baseline feature | `YHEA-600CA__2015` | Muscle-strengthening-activity status |
| Target | `YHEA-PAIN-1__2019` | 1 yes, 0 no; other raw codes are not additional pain classes |

For each activity status, distinguish **0 never**, **1 enter an amount**, and **2 unable**. These three status fields are **not minutes, frequency, or days per week**, and inability is not a higher activity dose. Activity may include work and other reasons as well as exercise. Consult the [health-practices guide](https://nlsinfo.org/content/cohorts/nlsy97/topical-guide/health/health-practices-knowledge) and [physical-health guide](https://nlsinfo.org/content/cohorts/nlsy97/topical-guide/health/physical-health).

**To complete:** justify coding general health as ordered or categorical. Encode activity status without imposing a false numerical ordering. If frequency/duration features are later added, document their units, skip rules, overlap, and changes to the configured contract.

### 2.3 Load through the package

The following cell locates the repository from either the repository root or its `notebooks/` directory, then uses the project's data boundary. It performs setup, not cohort preparation or model fitting. The current TOML contains an absolute data path; make that declaration portable before final clean-clone reproduction.


In [1]:
from pathlib import Path

from dsm_project.algorithms import registered_student_algorithms
from dsm_project.config import load_config
from dsm_project.contracts import TaskKind
from dsm_project.data import load_project_dataset

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "configs/nlsy97.toml").is_file()
    and (path / "pyproject.toml").is_file()
)
CONFIG_PATH = PROJECT_ROOT / "configs/nlsy97.toml"
TASK_KEY = "classification"
TRUSTED_REFERENCE = "logistic_regression"

config = load_config(CONFIG_PATH)
raw_dataset = load_project_dataset(config)
task = raw_dataset.tasks[TASK_KEY]
raw_frame = raw_dataset.observations.copy()
IDENTIFIER = raw_dataset.identifier
TARGET = task.target_column
FEATURES = list(task.feature_columns)
assert task.kind is TaskKind.CLASSIFICATION
assert TARGET is not None

print("Configuration:", CONFIG_PATH.relative_to(PROJECT_ROOT))
print("Raw source shape:", raw_frame.shape)
print("Configured predictors:", FEATURES)
print("Raw target:", TARGET)
print("Registered student algorithms:", registered_student_algorithms())
print("Setup only: no cleaned cohort, fitted model, or evaluation result yet.")


Configuration: configs/nlsy97.toml
Raw source shape: (8984, 147)
Configured predictors: ['CV_AGE_INT_DATE__2015', 'YHEA-100__2015', 'YHEA-600AA__2015', 'YHEA-600BA__2015', 'YHEA-600CA__2015']
Raw target: YHEA-PAIN-1__2019
Registered student algorithms: ('batch_linear_regression',)
Setup only: no cleaned cohort, fitted model, or evaluation result yet.


**Interpret the setup output.** Explain why source shape is not the eligible sample size. The CSV adapter checks structural requirements; it does not interpret survey missing codes, enforce a binary pain label, or clean the cohort. The TOML's `positive_event` text documents intent and does not recode the target. A successful load is therefore not permission to train on the raw frame.

**To complete:** describe the validation the package currently performs and the domain checks you still need to implement and test. Link to those package symbols once they exist.


## 3. Can the data support the question? Audit and purposeful exploration

### 3.1 Cohort, missingness, and information timing

**To complete before model fitting:**

1. Verify unique participant IDs, field types, documented values, interview availability, and baseline-before-outcome timing. Investigate repeated people, related household members, and other dependence relevant to partitioning.
2. Keep target labels 0 and 1; never impute an unavailable pain outcome. Use source-specific handling for −1 refusal, −2 don't know, −3 invalid skip, −4 valid skip, and −5 noninterview. Preserve reasons before choosing exclusions or imputation; a skipped item is not automatically zero.
3. Define baseline eligibility and missing-predictor handling. Keep unable-to-exercise distinct from never. Do not require a current job merely for an activity-only model; justify any job restriction for an occupational extension.
4. Produce an attrition table: source records → interviewed at baseline/follow-up → valid target → retained predictor/cohort rules. Report retained pain-positive and pain-negative counts, and examine selection effects.
5. Exclude identifiers, later health, pain frequency, survey weights, and future-updated fields from ordinary predictors unless a specific, justified role is documented. Restrict any combined cross-round field by its administration round.

**Protect evaluation data early.** Implement and record the eligible-cohort partition before exploratory modeling choices. Use development/training data for feature-outcome exploration and learned preprocessing. Keep the final evaluation outcomes sealed; Section 7 records the full evaluation protocol.

Reusable cohort construction and validation belong in tested package code. The following is a work slot, not an implemented preparation function.


In [ ]:
# TODO: Import tested cohort preparation and partitioning from dsm_project.
# TODO: Build the eligible cohort from raw_dataset without changing raw_frame.
# TODO: Check unique IDs, time order, binary labels, exclusions, and valid codes.
# TODO: Display the attrition table and document every exclusion.
# TODO: Save/reuse the same train/validation/final-test participant IDs.
# No cohort-preparation function or partition is supplied by this outline.


### 3.2 Purposeful development-data displays

For each display, write its question **before** the code and an interpretation **immediately after** it. Use human-readable labels and units.

| Display to produce | Question it answers | Decision it informs |
|---|---|---|
| Missing/special-code audit and attrition table | Which participants or fields become unavailable? | Exclusion, imputation, and scope |
| Development pain counts and prevalence | How imbalanced is the retained outcome? | Baseline and evaluation metrics |
| Age distribution and general-health categories | What ranges and category sizes are supported? | Encoding and extrapolation limits |
| Activity-status counts, including unable | Are categories sparse or being misinterpreted? | Encoding and stability checks |
| Development pain proportions by activity/health, with denominators and uncertainty | Is there overlap or a plausible predictive pattern? | Hypotheses and model limitations, not causal conclusions |

**To complete:** choose a small set of these displays, name their populations and denominators, and explain what each can and cannot show. Do not include a generic histogram gallery merely to fill space.


In [ ]:
# TODO: Produce only the tables/plots selected above, using development data.
# TODO: Label activity categories rather than plotting their codes as a dose.
# TODO: Report counts with proportions and label any survey-weighted estimates.
# Add a markdown interpretation immediately after each important display.


**Interpretation to write:** Which assumptions or modeling choices changed after exploration? Which patterns are uncertain, could reflect confounding/selection, or are unsupported outside this cohort?


## 4. Mathematical model: binary logistic regression

Use this as a teaching outline and supply the derivation and explanation, not just a list of equations. The [course logistic example](../examples/supervised/02-logistic-regression-and-decisions.ipynb) is one source to cite; add an authoritative source for any extension.

### 4.1 Notation and prediction

Define $n$ training participants and $d$ **encoded** features; $X\in\mathbb{R}^{n\times d}$, $x_i\in\mathbb{R}^d$, $y_i\in\{0,1\}$, coefficients $\beta\in\mathbb{R}^d$, and intercept $b\in\mathbb{R}$. Explain what every feature and $y_i=1$ mean in this study. The encoded dimension need not equal the five raw columns.

$$
z_i=b+x_i^\top\beta,\qquad
p_i=P(y_i=1\mid x_i)=\sigma(z_i)=\frac{1}{1+e^{-z_i}}.
$$

**To complete:** explain log-odds, the conditional linearity assumption, and why the model estimates a probability rather than a diagnosis. Distinguish this probability from the decision $\widehat y_i=\mathbf{1}[p_i\geq\tau]$. Threshold $\tau$ is a separate validation choice.

### 4.2 Objective, derivation, and optimization

Start from the average Bernoulli negative log-likelihood:

$$
J(\beta,b)=-\frac{1}{n}\sum_{i=1}^n
\left[y_i\log p_i+(1-y_i)\log(1-p_i)\right].
$$

**To complete:** derive the derivative with respect to $z_i$, then obtain the vector coefficient gradient and intercept gradient. Show the optimizer update used by your implementation, its learning rate, stopping rule, and initialization. Explain a numerically stable sigmoid and the equivalent stable loss $\log(1+e^{z_i})-y_i z_i$ without evaluating unsafe exponentials directly.

If you use regularization, class weights, or a different optimizer, explicitly write the changed objective and gradients, normalization, and intercept treatment. Do not compare a regularized reference to an unregularized student objective and assume identical coefficients should result.

### 4.3 Assumptions, fitted state, and limitations

**To complete:** explain conditional linear log-odds, dependence between participants, separation, collinearity, sparse categories, imbalance, and convergence problems. Predictors do not have to be normally distributed. Define fitted parameters, class order, prediction shapes, and the meaning of coefficients after your encoding/scaling. Discuss time and memory cost in terms of $n$, $d$, and optimization iterations. Explain why a coefficient or odds ratio here is not a causal effect.


In [ ]:
# TODO: Add a small explanatory calculation or plot if it helps teach the model.
# Example purpose: illustrate how the sigmoid converts a score to a probability.
# Do not define the reusable estimator, optimizer, or loss implementation here.


**Interpretation to write:** connect the mathematics to the data choices. What does the linear log-odds assumption miss, and which diagnostic could reveal that limitation?


## 5. Student implementation and its package interface

**Current starting point:** this checkout registers only the supplied `batch_linear_regression` exemplar. A student binary logistic estimator is not implemented or registered yet. The linear regressor is not a substitute for that classifier.

**To complete:** implement logistic regression under [the student package](../src/dsm_project/algorithms/student/), register it in [the algorithm registry](../src/dsm_project/algorithms/registry.py), and link the actual symbols and tests here once they exist. Choose the class and registry names yourself; this outline does not invent an implemented API.

Document:

- `fit`, `predict_proba`, and `predict`, their accepted inputs, and return types/shapes;
- fitted coefficients/intercept, feature count, class mapping, and convergence information;
- finite numeric inputs after preprocessing and binary target validation;
- probability bounds, class ordering, and the threshold convention;
- reproducibility controls, numerical safeguards, and errors for invalid input or use before fitting.

Reusable encoders, imputation, cohort logic, metrics, and partitioning also belong in `src/` or `scripts/` and tests. The notebook imports and explains that behavior.


In [ ]:
# TODO: Import create_student_algorithm and your tested preprocessing boundary.
# TODO: Record the real registry key after the logistic estimator is registered.
# TODO: Construct the student estimator/pipeline after verifying its interface.
# Run small verification fits in Section 6. Reserve project-cohort performance
# fitting and evaluation for Section 8, after the evaluation design is fixed.


## 6. Verification: does the implementation do what the mathematics says?

**To complete:** summarize inspectable tests and link their files and observed outcomes. Passing a test is implementation evidence, not evidence of useful health prediction.

| Verification | Evidence to supply |
|---|---|
| Hand-checkable or analytic case | Explain a tiny case with a known score, probability, loss, or update; compare observed output with the calculation. |
| Finite-difference gradient | Check coefficient and intercept gradients on nondegenerate test data; state the step size, tolerance, and observed error. |
| Probability and numerical properties | Valid shape/class order, probabilities in $[0,1]$, rows summing to 1 where applicable, stable extreme-score behavior, finite loss, reproducible fitting. |
| Failure boundaries | Before-fit use, nonbinary/missing labels, incompatible shapes, invalid hyperparameters, nonfinite features, and convergence failure. |
| Differential comparison | Compare the student estimator with a trusted classifier on matched inputs/objectives; explain normalization, regularization, intercept, tolerance, class order, and solver differences. |
| Data preparation and evaluation | Test negative survey codes, valid target filtering, non-ordinal activity encoding, unique IDs, partition separation, and training-only fitted transformations. |

The reference runner is **trusted-library code**, not student implementation. Its existence does not establish that your estimator passes these checks. Preserve failures and explain fixes or legitimate disagreements.


In [ ]:
# TODO: Import and demonstrate already-tested verification behavior.
# TODO: Report the exact test command, test files, tolerances, and actual result.
# TODO: Show a compact fair student-versus-reference comparison.
# Do not claim a passing check until you have run it.


**Interpretation to write:** Which checks passed, what remains unsupported, and how did you distinguish an implementation defect from a mismatch in reference conventions?


## 7. Evaluation design, baselines, and decision rules

### 7.1 Partitions and preprocessing

**To complete before training/tuning:** record eligible participant IDs and the training, validation/CV, and sealed final-test roles. The current configuration uses seed `438` and a `0.25` holdout fraction. Justify any change and your stratification/grouping policy. All features precede the outcome, but a random participant holdout does not demonstrate performance in a newly recruited future cohort. Assess household dependence; keep every repeated person in one partition if extending to repeated rows.

Fit imputation, category encoding, scaling, feature selection, and any resampling on training folds only. Specify how unknown categories and missing inputs are handled at prediction time. Tune regularization, thresholds, calibration, or other choices on development/validation data, then freeze them before one final evaluation.

**Existing runner limitation:** `evaluate_supervised_estimator` and `fit_supervised_candidate` currently make a single stratified random train/holdout split and report balanced accuracy. They do not implement train/validation/final separation, grouped/time splitting, CV, uncertainty, or arbitrary configured metrics. The built-in trusted logistic pipeline scales raw numeric inputs; it does not supply survey cleaning, imputation, or categorical encoding. Use it only when its behavior matches your declared experiment. Put needed extensions in tested package code; a TOML setting alone is not evidence that a rule is enforced.

### 7.2 Baselines and feature comparisons

Use the same eligible participants, partitions, preprocessing conventions, and evaluation rules for each incremental comparison.

| Candidate | Purpose |
|---|---|
| Majority/no-pain decision and training-prevalence probability | Minimum decision and probability baselines; estimate prevalence using training labels only. |
| Logistic: baseline age + general health | Establish performance before including physical activity. |
| Logistic: baseline age + general health + three activity statuses | Test the governing incremental-prediction question. |
| Trusted classifier matched to the student model | Check implementation agreement; keep implementation comparisons distinct from scientific feature comparisons. |
| Optional validated occupational-sitting extension | Add only after the code mapping and coverage are validated; compare on a common cohort or separate coverage changes from predictive gains. |

### 7.3 Metrics, threshold, and uncertainty

The configured primary metric is balanced accuracy. Define the confusion-matrix counts for pain as the positive event and explain:

$$
\operatorname{BalancedAccuracy}=\frac{1}{2}
\left(\frac{TP}{TP+FN}+\frac{TN}{TN+FP}\right).
$$

State its range, why it suits your investigation, and what it misses. Accuracy alone can hide poor pain detection. Choose justified secondary evidence such as precision/recall, precision–recall performance, log loss/Brier score, and calibration; distinguish probability quality from threshold-specific decisions. Define every metric actually reported, its direction/range, and relevant baseline. Specify how you handle undefined quantities and folds without enough events.

**To complete:** predeclare the primary comparison and uncertainty procedure. Use paired evaluation for models tested on the same people, preserve dependence when resampling, and report intervals or fold variability without treating them as causal or population guarantees. Do not select features, thresholds, or a preferred model using final-test results.


In [ ]:
# TODO: Import the tested partition, preprocessing, and evaluation components.
# TODO: Record split sizes/IDs, class counts, hyperparameters, and seed policy.
# TODO: Freeze the metric, baseline, threshold rule, and comparison plan.
# TODO: Verify that the actual runner enforces the plan before calling it.


## 8. Results, uncertainty, and practical meaning

Run project-cohort performance fitting and evaluation only after Sections 3–7 are implemented and verified; small test-data fits belong in Section 6. Label each output as **development**, **validation**, or **final-test** evidence and each estimator as **student-authored**, **trusted reference**, or **baseline**.

Before the code, state exactly which question the experiment answers. After it, interpret the result against the baseline and uncertainty; do not leave an unexplained score or plot.


In [ ]:
# TODO: Run the frozen experiment through tested dsm_project package code.
# Existing supervised API names (use only after their behavior matches the plan):
# from dsm_project.experiments import (
#     evaluate_supervised_estimator,
#     fit_supervised_candidate,
# )
# TODO: Supply a prepared dataset, not raw_dataset with survey missing codes.
# TODO: Fit student/reference/baseline candidates on identical partitions.
# TODO: Preserve settings, timings, candidate records, and prediction evidence.
# This outline intentionally performs no fitting.


### 8.1 Results to report

**To complete:** provide a compact table with implementation type, feature set, sample counts, evidence split, primary metric with uncertainty, justified secondary metrics, and runtime. Preserve losing candidates. Identify data/config/source versions, partition records, preprocessing, hyperparameters, convergence, and seed policy so the comparison can be reproduced.

### 8.2 Answer the incremental question

**Write here after running:** Does adding activity improve held-out prediction relative to age and general health alone? How large is the change, how uncertain is it, and would it matter for the stated investigation? Could any apparent gain reflect changed cohort composition or preprocessing? A negligible gain is a valid finding. Compare student/reference agreement separately; agreement does not prove useful prediction.

**Interpret every display immediately.** Do not populate this section with imagined model scores, the raw source row count, or previous illustrative cohort counts as if they were current results.


## 9. Failure analysis and sensitivity

**To complete:** choose diagnostics that challenge the model and relate errors to the intended use.

- Show confusion patterns at the declared threshold with counts and denominators. Explain missed pain reports and false alarms without treating predictions as diagnoses.
- Inspect calibration and probability distributions; confidence and correctness are different. Any threshold/calibration adjustment uses validation evidence, not final-test tuning.
- Examine performance by relevant health/activity categories, especially unable-to-exercise, and other justified slices. Report support and uncertainty; small slices cannot sustain strong fairness or subgroup claims.
- Investigate outliers, sparse categories, missingness/exclusion sensitivity, reasonable encoding/regularization changes, dependence, convergence, and resource cost. Explain whether each sensitivity analysis is exploratory or prespecified.
- Separate measurement/selection problems, task limitations, model assumptions, implementation defects, and distribution shift. Consider the changing interview period and limits of generalizing beyond this cohort.
- Explain which failures input/score monitoring could reveal and which require later observed pain labels. Link proposed monitoring to the [operations plan](../docs/OPERATIONS.md).

If O*NET is added, assess mapping ambiguity and missing coverage. Occupational sitting ratings are neither personal sitting hours nor a measure of stretching or posture. Do not multiply an ordinal rating by work hours and call the result measured sitting time.


In [ ]:
# TODO: Generate selected diagnostics from recorded validation/final predictions.
# TODO: Include counts, denominators, units, interval limitations, and split labels.
# TODO: Link any reusable diagnostic or sensitivity analysis to package code.
# Interpret each diagnostic in a following markdown cell.


**Failure interpretation to write:** identify the most consequential failure, the evidence for its cause, the proposed remedy, and what the current data cannot resolve. Preserve negative or inconclusive evidence.


## 10. Bounded conclusion and product decision

**To complete with observed evidence:**

1. Answer the opening question directly, including the estimated improvement or lack of improvement and uncertainty.
2. State the strongest supported claim and the most important limitation. Distinguish prediction from causation, later pain from new-onset pain, and cohort performance from general clinical validity.
3. Decide whether to reject the method, retain it as a baseline, investigate it further, or promote it. Explain the evidence supporting that choice and the next experiment most likely to change it.
4. State intended/prohibited uses and how a user should understand uncertainty.
5. If promotion is justified, link the tested preprocessing/model artifact and [model card](../docs/MODEL_CARD.md); explain the input contract, model identity, fallback, monitoring, and rollback implications. If not, explain what the product should do instead. Do not claim API/deployment work is complete merely because this notebook runs.

**Write here:** a concise answer paragraph, a limitations paragraph, and an explicit promotion/next-step decision. Update the [product contract](../docs/PRODUCT_CONTRACT.md) to match the actual scope.


## Completion checklist for this notebook

This outline follows the assignment, but a completed submission must replace its prompts with your own reasoning, implemented package calls, and observed evidence.

- [ ] One governing question, reader, target, population, prediction time, and responsible-use boundary are clear.
- [ ] Source/access/version, sampling, variable meanings, actual interview timing, and dataset-card links are complete.
- [ ] Cohort construction, missing-code handling, label validation, exclusions, dependence, and partitioning are tested and documented.
- [ ] Development exploration has stated questions, labeled figures/tables, and immediate interpretations.
- [ ] Notation, logistic objective, derivation, optimizer, stability, assumptions, hyperparameters, and cost are explained before fitting.
- [ ] The actual student estimator, registry key, public interface, and supporting tests are linked.
- [ ] Analytic/property/failure and finite-difference tests plus fair trusted comparisons have observed outcomes.
- [ ] Preprocessing/splits/baselines/metrics/thresholds/uncertainty match actual package behavior and preserve the sealed final test.
- [ ] Results distinguish student, trusted, and baseline code; retain candidate evidence; report uncertainty and reproducibility details.
- [ ] Failure analysis, limitations, responsible use, and the promotion/next-experiment decision answer the opening question.
- [ ] No unfinished prompts, undefined variables, inappropriate regression fallback, unsupported sitting feature, or invented results remain.
- [ ] No private absolute paths or hidden execution state remain in the finished analysis; keep raw local data out of Git.
- [ ] A fresh **DSM Project** kernel runs every cell top to bottom. Clear saved outputs before review unless the course asks otherwise.

Other final-project tasks, implementations, API/UI, tests, monitoring, CI, and release evidence are project-wide obligations. This classification notebook alone does not satisfy that entire scope.
